# 16 — Docker do serving de laboratório

A referência congelada foi exportada pelo autor e validada por HTTP em um wheel
instalado fora do checkout. Esta etapa empacota o mesmo runtime com dependências
fixadas, modelo externo somente para leitura e identidade verificada na carga.

Não há refit, acesso ao holdout ou consumo de 16–30/09. O produtor de features
causais e a fila de investigação continuam responsabilidades separadas.


## Resultado recebido e alcance

O exportador relatou 67.255 linhas de paridade. O wheel validou o caso HTTP do
evento 335910, com score 0.4178630794049542 e tolerância 1e-12. Release:
`b3deb7e7bb6d4189bc7671453f8643d2`; modelo:
`9728f8dddc4daa9e5478db1a8a173adecb71e41c5992965c117d2b46a27625d9`.

[Relato nativo](../../references/evidence/laboratory_serving_native_validation_2026-10-09.json):
o assistente recebeu saídas de terminal, sem os bytes do modelo/manifesto/wheel.
A construção e execução Docker ainda precisam acontecer no Alienware.


## Execução explícita no host

O [runbook de serving](../../docs/operations/SERVING_CONTRACT.md#construir-e-verificar-docker)
concentra os comandos, proteções, limites e recuperação. O build usa
`docker/serving/Dockerfile`; o verificador está em `scripts/serving/`.

```bash
make build-serving-image
make validate-serving-docker \
  RELEASE_PATH="$PWD/data/serving/reference-v1" \
  MANIFEST_SHA256="1dd762f7e25fde17ea83a00df5ab489e6885b85f0acd10a1a47da8804bed3263"
```

O recibo `data/serving/docker-check.json` deve ser novo. O check usa a imagem por ID,
confere HTTP/health/readiness, ausência da stack offline, root/modelo read-only,
reinício e falhas de SHA/modelo alterado/modelo ausente. Falhas interrompem o check;
nenhuma saída positiva é fabricada.


In [ ]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
report_path = root / "data/serving/docker-check.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    result = {
        key: report[key]
        for key in (
            "status",
            "image",
            "evidence_kind",
            "http_score",
            "restart_recovery",
            "failure_scenarios",
            "resources",
            "latency",
        )
    }
else:
    result = {"status": "awaiting_native_docker_execution", "report_path": str(report_path)}
result

## Interpretar as medições e avançar

Os percentis usam 100 requisições sequenciais ao mesmo caso depois de dez de
aquecimento. Incluem transporte HTTP local, não demonstram carga concorrente ou
SLA. A memória/CPU de `docker stats` é um snapshot; o pico de memória não é medido.
A base conserva Python 3.14.4; um digest verificado pode fixar seus bytes.

A preparação testa o verificador com HTTP sintético real e falhas controladas,
sem um engine Docker disponível. Apenas o recibo nativo permitirá fechar esse
marco. Depois de revisá-lo, seguimos para Prefect, replay causal e monitoramento,
conforme o [mural](../../docs/project/ROADMAP.md#próxima-entrega-concreta).
